# PhD-student wellbeing survey analysis

A standalone statistics project outside structural biology: scoring PHQ-9 and GAD-7 questionnaire responses from survey exports and comparing groups. Operates on external survey CSVs that are intentionally excluded from this repository.


In [ ]:
import pandas as pd
import numpy as np

# Load files
df_social_sci = pd.read_csv("data_mental_health_of_phd_sudents_2_2026-05-06_11-43_spss.csv", encoding='latin1', delimiter=';') # Changed encoding and added delimiter
df_natural_sci = pd.read_csv("data_mental_health_of_phd_sudents_2026-05-06_11-43_spss.csv", encoding='latin1', delimiter=';') # Changed encoding and added delimiter

phq_cols = [f"SC01_{i:02d}" for i in range(1, 10)]
gad_cols = [f"SC03_{i:02d}" for i in range(1, 8)]

In [ ]:
print(f"Number of rows in df_social_sci: {df_social_sci.shape[0]}")
print(f"Number of rows in df_natural_sci: {df_natural_sci.shape[0]}")

In [ ]:
df_social_sci["STATUS"].value_counts(dropna=False)
df_natural_sci["STATUS"].value_counts(dropna=False)

In [ ]:
def filter_completed(df):
    df = df.copy()

    # Remove first descriptive/question-label row if present
    # In your export, row 0 seems to contain labels like "Interview status..."
    if "STATUS" in df.columns:
        # Convert to string, strip whitespace, and convert to lowercase for robust comparison
        status_column_processed = df["STATUS"].astype(str).str.strip().str.lower()

        # Check for and remove the header row
        if status_column_processed.iloc[0].startswith("interview status"):
            df = df.iloc[1:].copy()
            # Re-process status_column_processed after dropping the row
            status_column_processed = df["STATUS"].astype(str).str.strip().str.lower()

        # Keep only completed interviews
        # Use .str.contains to handle variations like "complete questionnaire" or other substrings
        # Changed 'completed' to 'complete' based on value_counts output
        df = df[
            status_column_processed.str.contains("complete", na=False)
        ].copy()
    else:
        print("Warning: 'STATUS' column not found in DataFrame. Returning original df.")

    return df

df_social_sci_completed = filter_completed(df_social_sci)
df_natural_sci_completed = filter_completed(df_natural_sci)

print("Social sciences completed:", len(df_social_sci_completed))
print("Natural sciences completed:", len(df_natural_sci_completed))

In [ ]:
def score_mental_health(df, group_name=None):
    df = df.copy()

    phq_cols = [
        "SC01_01", "SC01_02", "SC01_03",
        "SC01_04", "SC01_05", "SC01_06",
        "SC01_07", "SC01_08", "SC01_09"
    ]

    gad_cols = [
        "SC03_01", "SC03_02", "SC03_03",
        "SC03_04", "SC03_05", "SC03_06", "SC03_07"
    ]

    # Convert to numeric
    df[phq_cols + gad_cols] = df[phq_cols + gad_cols].apply(
        pd.to_numeric, errors="coerce"
    )

    # Convert any values less than 0 (likely missing data codes) to NaN
    for col in phq_cols + gad_cols:
        df.loc[df[col] < 0, col] = np.nan

    # If PHQ is coded 1-4, recode to 0-3
    phq_min = df[phq_cols].min().min()
    phq_max = df[phq_cols].max().max()

    if phq_min >= 1 and phq_max <= 4:
        df[phq_cols] = df[phq_cols] - 1
        print(f"{group_name}: PHQ was 1-4, recoded to 0-3")
    else:
        print(f"{group_name}: PHQ not recoded")
        print("PHQ range:", phq_min, "to", phq_max)

    # Sum scores per row
    df["PHQ9_total"] = df[phq_cols].sum(axis=1)
    df["GAD7_total"] = df[gad_cols].sum(axis=1)

    # Combined score
    df["PHQ_GAD_total"] = df["PHQ9_total"] + df["GAD7_total"]

    # Optional group label
    if group_name is not None:
        df["science_group"] = group_name

    # Tracking outputs
    print("\nScore summary:")
    print(df[["PHQ9_total", "GAD7_total", "PHQ_GAD_total"]].describe())

    print("\nFirst rows:")
    print(df[["PHQ9_total", "GAD7_total", "PHQ_GAD_total"]].head())

    return df

In [ ]:
sc_columns = [col for col in df_natural_sci_completed.columns if 'SC' in col]
display(df_natural_sci_completed[sc_columns])

In [ ]:
print("Social sci rows with -9:")
print(df_social_sci.index[df_social_sci[phq_cols].eq(-9).any(axis=1)].tolist())

print("Natural sci rows with -9:")
print(df_natural_sci.index[df_natural_sci[phq_cols].eq(-9).any(axis=1)].tolist())

In [ ]:
rows_with_minus_9 = df_natural_sci_completed.index[df_natural_sci_completed[phq_cols].eq(-9).any(axis=1)].tolist()

print(f"Shape of df_natural_sci_completed before removing rows: {df_natural_sci_completed.shape}")

df_natural_sci_completed = df_natural_sci_completed.drop(rows_with_minus_9)

print(f"Shape of df_natural_sci_completed after removing rows: {df_natural_sci_completed.shape}")

In [ ]:

rows_with_minus_9_soc = df_social_sci_completed.index[df_social_sci_completed[phq_cols].eq(-9).any(axis=1)].tolist()

print(f"Shape of df_social_sci_completed before removing rows: {df_social_sci_completed.shape}")

df_social_sci_completed = df_social_sci_completed.drop(rows_with_minus_9_soc)

print(f"Shape of df_natural_sci_completed after removing rows: {df_social_sci_completed.shape}")

In [ ]:
# Score both datasets
df_social_sci_scored = score_mental_health(df_social_sci_completed, group_name="social_sciences")
df_natural_sci_scored = score_mental_health(df_natural_sci_completed, group_name="natural_sciences")

# Combine into one dataset for later group comparison
df_all = pd.concat(
    [df_social_sci_scored, df_natural_sci_scored],
    ignore_index=True
)
